In [2]:
import pandas as pd
import numpy as np
from datetime import datetime

### Tabela Music_Info

In [187]:
music_info = pd.read_csv('music_info.csv', sep=None, engine="python", dtype=str, keep_default_na=True)

In [188]:
# remove BOM no início dos nomes de coluna (se houver)
music_info.columns = music_info.columns.str.replace(r'^\ufeff', '', regex=True)

In [ ]:
cols = ['track_id','name','artist','genre','year','danceability','energy']
missing = [c for c in cols if c not in music_info.columns]
if missing:
    print(f"Colunas ausentes no music_info: {missing}")
music_info = music_info.reindex(columns=cols)

In [191]:
music_info['danceability'] = pd.to_numeric(music_info['danceability'], errors='coerce')
music_info['energy'] = pd.to_numeric(music_info['energy'], errors='coerce')

In [192]:
music_info['genre'].unique()

array([nan, 'RnB', 'Rock', 'Pop', 'Metal', 'Electronic', 'Jazz', 'Punk',
       'Country', 'Folk', 'Reggae', 'Rap', 'Blues', 'New Age', 'Latin',
       'World'], dtype=object)

In [193]:
cols = ['name', 'artist', 'genre']
for col in cols:
    music_info[col] = music_info[col].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)
    music_info[col] = music_info[col].replace({'': pd.NA, 'nan': pd.NA})

In [ ]:
# Converte para inteiro anulável (preserva NaN)
music_info['year'] = music_info['year'].astype(str).str.strip().replace({'': pd.NA})
music_info['year'] = pd.to_numeric(music_info['year'], errors='coerce').astype('Int64')

###  Tabela User_Listening


In [3]:
user_listening = pd.read_csv('user_listening.csv', sep=None, engine="python", dtype=str, keep_default_na=True)

In [4]:
# Remove BOM no início dos nomes de coluna (se houver)

user_listening.columns = user_listening.columns.str.replace(r'^\ufeff', '', regex=True)

In [5]:
# Garante playcount numérico

user_listening['playcount'] = pd.to_numeric(
    user_listening['playcount'].astype(str).str.strip(),
    errors='coerce'
).fillna(0).astype(int)

In [6]:
# Agrupa por user_id e reduz linhas somando playcounts
user_playcounts = (
    user_listening
    .groupby('user_id', as_index=False)
    .agg(
        total_playcount=('playcount', 'sum'),

    )
)

# Ajustes de tipo/opções
user_playcounts['total_playcount'] = user_playcounts['total_playcount'].astype('Int64')

# Checagens rápidas
print('Linhas originais user_listening:', len(user_listening))
print('Linhas agregadas (usuários):', len(user_playcounts))
print('Soma playcounts original:', int(user_listening['playcount'].sum()))
print('Soma playcounts agregado:', int(user_playcounts['total_playcount'].sum()))

# 5) (Opcional) salvar resultado
# user_playcounts.to_csv('user_playcounts_by_user.csv', index=False)

Linhas originais user_listening: 9711301
Linhas agregadas (usuários): 962037
Soma playcounts original: 25549912
Soma playcounts agregado: 25549912


In [7]:
# Pegando 100000 usuários com mais plays

# quantos usuários queremos
top_n = 1000

# ordena desc e pega os top N
top_user_playcounts = (
    user_playcounts
    .sort_values(by='total_playcount', ascending=False)
    .head(top_n)
    .reset_index(drop=True)
)

# lista de user_ids selecionados (strings)
top_user_ids = top_user_playcounts['user_id'].astype(str).tolist()

print(f"Selecionados {len(top_user_ids)} user_ids (top {top_n})")


Selecionados 1000 user_ids (top 1000)


In [8]:
# Torna set de strings para busca rápida e robusta a tipos
top_user_set = set(map(str, top_user_ids))

# Normaliza user_id no dataframe e filtra
user_listening['user_id'] = user_listening['user_id'].astype(str).str.strip()
user_listening_filtered = user_listening[user_listening['user_id'].isin(top_user_set)].reset_index(drop=True)

# Checagem rápida
print(f"Linhas originais user_listening: {len(user_listening)}")
print(f"Linhas após filtro (top users): {len(user_listening_filtered)}")



Linhas originais user_listening: 9711301
Linhas após filtro (top users): 84296


In [9]:
user_listening = user_listening_filtered

In [ ]:
import pandas as pd
df = pd.read_csv("user_listening_export.csv")
df['playcount'] = df['playcount'].astype(int)
df = df.sort_values(['user_id','playcount'], ascending=[True, False])
topN = df.groupby('user_id').head(20)   # ajuste o 20
topN.to_csv("user_listening_top20.csv", index=False)


### Tabela User_Profiles

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

# assume `users_filtered` exists and has a 'user_id' column
user_ids = user_listening['user_id'].dropna().astype(str).unique()
n = len(user_ids)

rng = np.random.default_rng(42)

# example name parts / countries (adjust lists as desired)
first_names = ['alex','maria','john','ana','david','laura','michael','sofia','chris','mariana',
               'james','isabela','robert','camila','daniel','carla','paul','juliana','mark','daniela', 'lulu', 'luca']
last_names = ['silva','souza','smith','johnson','costa','pereira','brown','oliveira','miller','alves',
              'martins','gomes','clark','rodrigues','lewis','fernandes','walker','ribeiro','hall','rocha']
countries = ['United States','Brazil','United Kingdom','Canada','Australia','Germany','France','Spain','Italy','Mexico',
             'Japan','South Korea','Netherlands','Sweden','Argentina','Portugal','Colombia','Chile','Peru','South Africa']

# generate readable unique usernames
usernames = [f"{rng.choice(first_names)}.{rng.choice(last_names)}{i:04d}" for i in range(n)]

# demographics
ages = rng.integers(13, 81, size=n)                      # 13..80
sex = rng.choice(['M','F','Other'], size=n, p=[0.49, 0.49, 0.02])
country = rng.choice(countries, size=n)

# signup_date: random between 2010-01-01 and today
start = pd.to_datetime('2010-01-01')
end = pd.to_datetime(datetime.utcnow().date())
days_range = (end - start).days
signup_offsets = rng.integers(0, days_range + 1, size=n)
signup_date = (start + pd.to_timedelta(signup_offsets, unit='D')).normalize()

# subscription_type
subscription_type = rng.choice(['free', 'premium'], size=n, p=[0.7, 0.3])

# last_active: random date between signup_date and today for each user
last_active = []
for sd in signup_date:
    max_days = max((end - sd).days, 0)
    offset = 0 if max_days == 0 else int(rng.integers(0, max_days + 1))
    last_active.append((sd + pd.to_timedelta(offset, unit='D')).normalize())

# assemble dataframe
user_profiles = pd.DataFrame({
    'user_id': user_ids,
    'username': usernames,
    'age': ages,
    'sex': sex,
    'country': country,
    'signup_date': signup_date,
    'subscription_type': subscription_type,
    'last_active': pd.to_datetime(last_active)
})

# reset index and show a quick summary
user_profiles = user_profiles.reset_index(drop=True)
print('Generated user_profiles rows:', len(user_profiles))
user_profiles.head(5)

Generated user_profiles rows: 5000


,user_id,username,age,sex,country,signup_date,subscription_type,last_active
0,5a905f000fc1ff3df7ca807d57edb608863db05d,maria.fernandes0000,36,F,Peru,2013-09-06,free,2023-08-07
1,8814f5d1f1d7177aa2efb6de6454504f3bb7b7bc,daniel.miller0001,27,F,Argentina,2015-07-10,premium,2021-01-18
2,ea64e003562d2f0f39e5a7dd84af5b1969e0fea3,mariana.ribeiro0002,48,M,Italy,2023-09-03,free,2025-06-10
3,ec25e3d78ea8374869a772dc58bb903528a3c9cc,maria.rodrigues0003,58,M,Brazil,2011-08-20,free,2014-07-18
4,b12c786deef0e618b5f277bc337f67128f425efe,david.souza0004,30,F,Canada,2011-09-22,free,2015-07-10


### Data Transformation

In [205]:
# resultado: DataFrame `music_info_with_play` com todas as colunas originais + `total_playcount`

# limpeza básica dos ids (remove espaços invisíveis)
music_info['track_id'] = music_info['track_id'].astype(str).str.strip()
user_listening['track_id'] = user_listening['track_id'].astype(str).str.strip()

# garantir playcount numérico (coerce -> NaN -> 0)
user_listening['playcount'] = pd.to_numeric(
    user_listening['playcount'].astype(str).str.strip(),
    errors='coerce'
).fillna(0).astype(int)

# agregação: total de plays por track_id
playcounts = (
    user_listening
    .groupby('track_id', as_index=False)['playcount']
    .sum()
    .rename(columns={'playcount': 'total_playcount'})
)

# merge com music_info (mantém todas as tracks do music_info)
music_info_with_play = music_info.merge(playcounts, on='track_id', how='left')

# preencher NaN (tracks sem plays) com 0 e escolher tipo inteiro anulável
music_info_with_play['total_playcount'] = music_info_with_play['total_playcount'].fillna(0).astype('Int64')



In [207]:
# Filtra apenas tracks com total_playcount > 0
music_info_with_play_positive = music_info_with_play[
    music_info_with_play['total_playcount'].fillna(0) > 0
].reset_index(drop=True)

# checagem rápida (opcional)
music_info = music_info_with_play_positive

In [211]:
music_info.drop(columns=['total_playcount'], inplace=True, errors='ignore')

### Gerando os CSVs

In [213]:
music_info.to_csv('music_info_export.csv', sep=',', encoding='utf-8', index=False)
print("Saved `music_info` -> music_info_export.csv")

Saved `music_info` -> music_info_export.csv


In [10]:
user_listening.to_csv('user_listening_export.csv', sep=',', encoding='utf-8', index=False)
print("Saved `user_listening` -> user_listening_export.csv")

Saved `user_listening` -> user_listening_export.csv


In [215]:
user_profiles.to_csv('user_profiles_export.csv', sep=',', encoding='utf-8', index=False)
print("Saved user profiles -> user_profiles_export.csv")

Saved user profiles -> user_profiles_export.csv
